# BIS 438 - Part 2: Model Training Lab

**TeleConnect Customer Churn Prediction**

---

## Welcome Back

In **Part 1**, you cleaned and prepared customer data for machine learning. You:
- Handled missing values
- Removed duplicates
- Created new features
- Encoded categorical variables
- Split data into training and test sets

In **Part 2**, you'll use that prepared data to:
- Train three different prediction models
- Compare their performance
- Identify what drives customer churn
- Make predictions on new customers
- Deploy a model for production use

---

## The Business Problem (Reminder)

**TeleConnect** is losing customers to competitors. The company wants to:
1. **Predict** which customers are likely to leave (churn)
2. **Understand** what causes customers to leave
3. **Take action** by offering retention deals to at-risk customers

**Why this matters:** Acquiring a new customer costs 5-25x more than retaining an existing one.

---

## Learning Objectives

By the end of this lab, you'll understand:
- How machine learning models make predictions
- How to measure model accuracy
- What makes a "good" prediction model
- How to interpret model results for business decisions
- What overfitting means and why it matters

---

## Getting Started

**Instructions:**
- Read each section carefully
- Run cells in order from top to bottom
- Press **Shift + Enter** to run each cell
- Pay attention to visualizations and interpretations

Begin by running the first code cell below.

---

# Section 1: Load Prepared Data

In Part 1, you saved four files:
- `X_train.pkl` - Training features (960 customers)
- `X_test.pkl` - Test features (240 customers)
- `y_train.pkl` - Training labels (who actually churned)
- `y_test.pkl` - Test labels (who actually churned)

**Why split the data?**
- **Training set:** Used to teach the model patterns
- **Test set:** Used to evaluate how well the model works on *new* customers it's never seen

This simulates real-world deployment: the model will predict churn for customers it hasn't encountered before.

Let's load everything!

In [ ]:
# Import libraries needed for this lab

# pandas: Data manipulation and analysis (working with dataframes)
import pandas as pd

# numpy: Numerical computing and array operations
import numpy as np

# matplotlib: Creating visualizations and charts
import matplotlib.pyplot as plt

# seaborn: Statistical data visualization (built on matplotlib)
import seaborn as sns

# pickle: Serializing and deserializing Python objects (loading saved data)
import pickle

# os: Operating system interface (checking file sizes)
import os

# LogisticRegression: A linear model for binary classification (calculates churn probability)
from sklearn.linear_model import LogisticRegression

# DecisionTreeClassifier: Tree-based model that creates decision rules
from sklearn.tree import DecisionTreeClassifier

# RandomForestClassifier: Ensemble of decision trees for robust predictions
from sklearn.ensemble import RandomForestClassifier

# Classification metrics for model evaluation
# confusion_matrix: Shows true positives, false positives, true negatives, false negatives
# accuracy_score: Calculates percentage of correct predictions
# precision_score: Calculates trustworthiness of positive predictions (TP / (TP + FP))
# recall_score: Calculates ability to find all positives (TP / (TP + FN))
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score

# warnings: Suppress non-critical warning messages for cleaner output
import warnings
warnings.filterwarnings('ignore')

# Set visualization style for consistent, professional-looking charts
sns.set_style('whitegrid')  # White background with grid lines
plt.rcParams['figure.figsize'] = (10, 6)  # Default figure size: 10 inches wide, 6 inches tall

print("Libraries loaded successfully!")

In [ ]:
# Load the prepared data from Part 1
# pickle.load() deserializes the binary .pkl files back into Python objects

# Load training features (960 customers x features)
# X_train contains all customer attributes (tenure, monthly charges, services, etc.)
with open('X_train.pkl', 'rb') as f:  # 'rb' = read binary mode
    X_train = pickle.load(f)

# Load test features (240 customers x features)
# X_test contains the same structure as X_train but for different customers
with open('X_test.pkl', 'rb') as f:
    X_test = pickle.load(f)

# Load training labels (960 customers)
# y_train contains 0 (stayed) or 1 (churned) for each training customer
with open('y_train.pkl', 'rb') as f:
    y_train = pickle.load(f)

# Load test labels (240 customers)
# y_test contains the true outcomes we'll use to evaluate model performance
with open('y_test.pkl', 'rb') as f:
    y_test = pickle.load(f)

print("Data loaded successfully!")

# Display dataset dimensions
# .shape returns (rows, columns) tuple
print(f"\nTraining set: {X_train.shape[0]} customers, {X_train.shape[1]} features")
print(f"Test set: {X_test.shape[0]} customers, {X_test.shape[1]} features")

# Calculate churn rate (percentage who churned)
# .mean() on binary data gives the proportion of 1s
print(f"\nTraining set churn rate: {y_train.mean():.1%}")
print(f"Test set churn rate: {y_test.mean():.1%}")

### What We See

The data is ready for modeling:
- **960 training customers** to learn from
- **240 test customers** to evaluate performance
- **~41% churn rate** in both sets (good - balanced split!)
- All features are numeric and preprocessed

Now let's start building models!

---

# Section 2: Establish a Baseline

Before building complex models, let's answer a simple question:

**"What if we just predicted that EVERYONE stays (no one churns)?"**

This is called a **baseline model** - the simplest possible approach. Any real model should beat this!

## Why Start with a Baseline?

- It gives us a minimum performance target
- It shows whether our ML models are actually useful
- It's a reality check: sometimes simple rules work surprisingly well!

Let's calculate the baseline accuracy.

In [ ]:
# Create baseline predictions: predict everyone stays (churn = 0)
# np.zeros() creates an array of zeros with the same length as y_test
# This represents predicting "no churn" for every single customer
baseline_predictions = np.zeros(len(y_test))  # All zeros = no one churns

# Calculate how accurate this naive approach would be
# accuracy_score() compares predictions to actual outcomes
# Formula: (correct predictions) / (total predictions)
baseline_accuracy = accuracy_score(y_test, baseline_predictions)

print("BASELINE MODEL: 'Everyone Stays'")
print("="*50)
print(f"Accuracy: {baseline_accuracy:.1%}")  # .1% formats as percentage with 1 decimal

print(f"\nInterpretation:")
print(f"   If we predict 'no churn' for all {len(y_test)} customers,")
print(f"   we'd be correct {baseline_accuracy:.1%} of the time.")

# Calculate how many actual churners we'd miss
# y_test.sum() counts the number of 1s (churners) in the test set
print(f"\nProblem: We'd miss ALL {int(y_test.sum())} customers who actually churned!")

print(f"\nOur ML models need to beat {baseline_accuracy:.1%} accuracy AND identify churners.")

### Key Insight

The baseline gives us ~59% accuracy just by predicting "no churn" for everyone!

**But this is useless for business because:**
- We don't identify ANY at-risk customers
- We can't take preventive action
- We lose customers we could have saved

**We need a model that:**
- Has higher overall accuracy
- Actually identifies customers who will churn
- Minimizes false predictions

Let's build some real models!

---

# Section 3: Train Three Models

We'll train three different types of models and compare them:

## 1. Logistic Regression
- **How it works:** Calculates probability of churn based on weighted features
- **Strengths:** Simple, fast, interpretable
- **Think of it as:** A sophisticated scoring system

## 2. Decision Tree
- **How it works:** Creates a flowchart of yes/no questions about features
- **Strengths:** Easy to visualize, handles non-linear patterns
- **Think of it as:** A game of "20 questions" to guess churn

## 3. Random Forest
- **How it works:** Creates many decision trees and combines their predictions
- **Strengths:** Very accurate, handles complex patterns
- **Think of it as:** A committee of experts voting on the prediction

Let's train all three!

In [ ]:
print("Training models...\n")

# Model 1: Logistic Regression
# Creates a linear model that estimates churn probability using feature weights
# random_state=42: Sets random seed for reproducibility (same results each run)
# max_iter=1000: Maximum iterations for the optimization algorithm to converge
log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X_train, y_train)  # .fit() trains the model on training data
print("Logistic Regression trained")

# Model 2: Decision Tree
# Creates a tree of decision rules based on feature values
# random_state=42: Ensures consistent tree structure across runs
# max_depth=5: Limits tree to 5 levels deep (prevents overfitting)
decision_tree = DecisionTreeClassifier(random_state=42, max_depth=5)
decision_tree.fit(X_train, y_train)  # Learns optimal split points for each feature
print("Decision Tree trained")

# Model 3: Random Forest
# Ensemble method: creates multiple decision trees and averages their predictions
# random_state=42: Reproducible results
# n_estimators=100: Number of trees in the forest (more trees = more robust)
# max_depth=10: Each tree can be up to 10 levels deep
random_forest = RandomForestClassifier(random_state=42, n_estimators=100, max_depth=10)
random_forest.fit(X_train, y_train)  # Trains all 100 trees on random subsets of data
print("Random Forest trained")

print("\nAll models trained successfully!")

### Models Trained!

Each model has now learned patterns from the 960 training customers. 

**What happened behind the scenes?**
- **Logistic Regression:** Calculated optimal weights for each feature
- **Decision Tree:** Built a 5-level decision flowchart
- **Random Forest:** Created 100 different decision trees and learned how to combine them

Now let's see how well they perform!

---

# Section 4: Evaluate Model Performance

Now comes the critical question: **How good are our models?**

We'll measure performance using:

## Key Metrics

### 1. Confusion Matrix
Shows the four possible outcomes:
- **True Positives (TP):** Correctly predicted churn
- **True Negatives (TN):** Correctly predicted stayed
- **False Positives (FP):** Predicted churn but customer stayed (false alarm)
- **False Negatives (FN):** Predicted stayed but customer churned (missed opportunity)

### 2. Accuracy
**Formula:** (TP + TN) / Total
- Percentage of all predictions that were correct

### 3. Precision
**Formula:** TP / (TP + FP)
- Of customers we predicted would churn, what % actually did?
- **Business meaning:** How trustworthy are our churn predictions?

### 4. Recall
**Formula:** TP / (TP + FN)
- Of customers who actually churned, what % did we catch?
- **Business meaning:** How many churners did we successfully identify?

Let's calculate these for all three models!

In [ ]:
# Make predictions with all three models on the test set
# .predict() returns binary predictions (0 or 1) for each customer
log_reg_pred = log_reg.predict(X_test)
decision_tree_pred = decision_tree.predict(X_test)
random_forest_pred = random_forest.predict(X_test)

# Calculate performance metrics for each model
# We store everything in a dictionary for easy comparison
models_performance = {
    'Logistic Regression': {
        'predictions': log_reg_pred,
        # accuracy_score: (TP + TN) / Total
        'accuracy': accuracy_score(y_test, log_reg_pred),
        # precision_score: TP / (TP + FP) - "Of predicted churners, how many actually churned?"
        'precision': precision_score(y_test, log_reg_pred),
        # recall_score: TP / (TP + FN) - "Of actual churners, how many did we catch?"
        'recall': recall_score(y_test, log_reg_pred)
    },
    'Decision Tree': {
        'predictions': decision_tree_pred,
        'accuracy': accuracy_score(y_test, decision_tree_pred),
        'precision': precision_score(y_test, decision_tree_pred),
        'recall': recall_score(y_test, decision_tree_pred)
    },
    'Random Forest': {
        'predictions': random_forest_pred,
        'accuracy': accuracy_score(y_test, random_forest_pred),
        'precision': precision_score(y_test, random_forest_pred),
        'recall': recall_score(y_test, random_forest_pred)
    }
}

# Create a comparison table using pandas DataFrame
# This makes it easy to see which model performs best
comparison_df = pd.DataFrame({
    'Model': models_performance.keys(),
    'Accuracy': [m['accuracy'] for m in models_performance.values()],
    'Precision': [m['precision'] for m in models_performance.values()],
    'Recall': [m['recall'] for m in models_performance.values()]
})

# Display the comparison table
print("MODEL PERFORMANCE COMPARISON")
print("="*70)
print(comparison_df.to_string(index=False))  # index=False removes row numbers
print("="*70)

### Visualize Performance Comparison

In [ ]:
# Create a grouped bar chart to compare model performance across metrics
fig, ax = plt.subplots(figsize=(12, 6))  # Create figure and axis objects

# Set up bar positions
# np.arange() creates evenly spaced values for x-axis positions
x = np.arange(len(comparison_df))  # [0, 1, 2] for three models
width = 0.25  # Width of each bar (narrow enough for three bars per model)

# Create three sets of bars (one for each metric)
# x - width: shifts accuracy bars to the left
# x: centers precision bars
# x + width: shifts recall bars to the right
ax.bar(x - width, comparison_df['Accuracy'], width, label='Accuracy', color='skyblue')
ax.bar(x, comparison_df['Precision'], width, label='Precision', color='lightcoral')
ax.bar(x + width, comparison_df['Recall'], width, label='Recall', color='lightgreen')

# Customize the chart
ax.set_xlabel('Model', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)  # Set tick positions
ax.set_xticklabels(comparison_df['Model'])  # Label each position with model name
ax.legend()  # Show legend for colors
ax.set_ylim([0, 1])  # Set y-axis range from 0 to 1 (0% to 100%)

# Add baseline reference line
# axhline draws a horizontal line across the entire plot
ax.axhline(y=baseline_accuracy, color='red', linestyle='--', 
           label=f'Baseline ({baseline_accuracy:.1%})')

ax.grid(axis='y', alpha=0.3)  # Add subtle horizontal grid lines

plt.tight_layout()  # Adjust spacing to prevent label cutoff
plt.show()

print("\nAll models beat the baseline!")

### Confusion Matrices for Each Model

In [ ]:
# Create side-by-side confusion matrices for all three models
# This allows easy visual comparison of prediction patterns
fig, axes = plt.subplots(1, 3, figsize=(15, 4))  # 1 row, 3 columns

# Loop through each model and create its confusion matrix
for idx, (model_name, model_data) in enumerate(models_performance.items()):
    # Calculate confusion matrix: 2x2 grid of prediction outcomes
    # Returns [[TN, FP], [FN, TP]]
    cm = confusion_matrix(y_test, model_data['predictions'])
    
    # Create heatmap visualization
    # annot=True: Display numbers in each cell
    # fmt='d': Format numbers as integers (not scientific notation)
    # cmap='Blues': Color scheme (darker blue = higher count)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], 
                xticklabels=['Stayed', 'Churned'],  # Column labels
                yticklabels=['Stayed', 'Churned'])  # Row labels
    
    # Add title showing model name and accuracy
    axes[idx].set_title(f'{model_name}\nAccuracy: {model_data["accuracy"]:.1%}', 
                        fontweight='bold')
    
    # Label axes
    axes[idx].set_ylabel('Actual', fontweight='bold')  # What actually happened
    axes[idx].set_xlabel('Predicted', fontweight='bold')  # What model predicted

plt.tight_layout()  # Prevent overlap
plt.show()

### How to Read Confusion Matrices

Each matrix shows:
- **Top-left:** True Negatives (correctly predicted "stayed")
- **Top-right:** False Positives (predicted "churn" but actually stayed)
- **Bottom-left:** False Negatives (predicted "stayed" but actually churned) - Most costly!
- **Bottom-right:** True Positives (correctly predicted "churn") - Goal!

### Business Impact

**False Negatives (Bottom-left) are most expensive:**
- We think the customer will stay, so we do nothing
- Customer actually churns
- We lose the revenue and missed the chance to save them

**False Positives (Top-right) waste resources:**
- We offer retention deals to customers who weren't leaving anyway
- Less critical than missing real churners, but still costs money

---

# Section 5: Check for Overfitting

**Overfitting** is when a model performs great on training data but poorly on new data.

## Why This Matters

Imagine a student who:
- Memorizes all practice exam answers perfectly
- But can't solve slightly different problems on the real exam

That's overfitting! The model **memorized** training examples instead of learning general patterns.

## How to Detect Overfitting

Compare training vs. test performance:
- **Good:** Similar performance on both = model learned general patterns
- **Overfitting:** Much better on training than test = model memorized
- **Underfitting:** Poor on both = model too simple

Let's check our models!

In [ ]:
# Calculate training accuracy for each model to compare with test accuracy
# High training accuracy + low test accuracy = overfitting

# Generate predictions on the training set (same data models were trained on)
log_reg_train_pred = log_reg.predict(X_train)
decision_tree_train_pred = decision_tree.predict(X_train)
random_forest_train_pred = random_forest.predict(X_train)

# Create overfitting analysis dataframe
overfitting_df = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree', 'Random Forest'],
    # Training accuracy: how well model performs on data it was trained on
    'Training Accuracy': [
        accuracy_score(y_train, log_reg_train_pred),
        accuracy_score(y_train, decision_tree_train_pred),
        accuracy_score(y_train, random_forest_train_pred)
    ],
    # Test accuracy: how well model performs on new, unseen data (from earlier)
    'Test Accuracy': [
        models_performance['Logistic Regression']['accuracy'],
        models_performance['Decision Tree']['accuracy'],
        models_performance['Random Forest']['accuracy']
    ]
})

# Calculate the gap between training and test accuracy
# Large gap indicates overfitting (model memorized training data)
overfitting_df['Gap'] = overfitting_df['Training Accuracy'] - overfitting_df['Test Accuracy']

print("OVERFITTING ANALYSIS")
print("="*70)
print(overfitting_df.to_string(index=False))
print("="*70)

# Provide interpretation guidelines
print("\nInterpreting the Gap:")
print("   • Gap < 0.05 (5%): Good! Model generalizes well")
print("   • Gap 0.05-0.10: Acceptable, slight overfitting")
print("   • Gap > 0.10: Concerning, significant overfitting")

In [ ]:
# Visualize training vs. test accuracy to spot overfitting visually
fig, ax = plt.subplots(figsize=(10, 6))

# Set up grouped bar chart
x = np.arange(len(overfitting_df))  # Position for each model
width = 0.35  # Bar width

# Create two bars for each model: training and test accuracy
# Side-by-side comparison makes gaps obvious
ax.bar(x - width/2, overfitting_df['Training Accuracy'], width, 
       label='Training Accuracy', color='lightblue', alpha=0.8)
ax.bar(x + width/2, overfitting_df['Test Accuracy'], width, 
       label='Test Accuracy', color='lightcoral', alpha=0.8)

# Customize chart appearance
ax.set_xlabel('Model', fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax.set_title('Training vs. Test Accuracy (Overfitting Check)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(overfitting_df['Model'], rotation=15, ha='right')  # Rotate labels for readability
ax.legend()
ax.set_ylim([0.6, 1.0])  # Expanded range to show all values clearly
ax.grid(axis='y', alpha=0.3)  # Horizontal grid lines for easier reading

plt.tight_layout()
plt.show()

### What We Learned

Look at the gap between training and test accuracy:

- **Small gap:** Model learned general patterns (good!)
- **Large gap:** Model memorized training data (problematic!)

**Decision Trees** tend to overfit more than other models because they can create very specific rules for training data.

**Random Forests** usually handle this better by combining many trees.

**For production deployment,** we want a model that generalizes well to new customers!

---

# Section 6: Feature Importance - What Drives Churn?

Now for one of the most valuable insights: **What actually causes customers to churn?**

## Why This Matters for Business

Understanding feature importance helps TeleConnect:
- **Fix root causes:** Address the biggest churn drivers
- **Target interventions:** Focus retention efforts where they matter most
- **Improve products:** Enhance features that keep customers happy
- **Allocate resources:** Invest in what actually impacts retention

We'll use the **Random Forest** model because it's typically most accurate and provides reliable importance scores.

Let's see what matters most!

In [ ]:
# Extract feature importance from Random Forest model
# Random Forest calculates importance by measuring how much each feature
# reduces prediction error across all trees in the forest

feature_importance = pd.DataFrame({
    'Feature': X_train.columns,  # Feature names from training data
    # .feature_importances_ is an attribute created during model training
    # Higher value = more important for predictions
    'Importance': random_forest.feature_importances_
}).sort_values('Importance', ascending=False)  # Sort from most to least important

# Show top 10 most important features
# These are the strongest predictors of customer churn
top_10_features = feature_importance.head(10)

print("TOP 10 CHURN DRIVERS")
print("="*70)
print(top_10_features.to_string(index=False))
print("="*70)

In [ ]:
# Visualize feature importance using horizontal bar chart
# Horizontal layout makes feature names easier to read
plt.figure(figsize=(12, 8))

# Create horizontal bars
# range(len()) creates positions [0, 1, 2, ... 9] for 10 features
plt.barh(range(len(top_10_features)), top_10_features['Importance'], color='steelblue')

# Set y-axis labels to feature names
plt.yticks(range(len(top_10_features)), top_10_features['Feature'])

# Label axes
plt.xlabel('Importance Score', fontsize=12, fontweight='bold')
plt.ylabel('Feature', fontsize=12, fontweight='bold')
plt.title('Top 10 Features Predicting Customer Churn', fontsize=14, fontweight='bold')

# Invert y-axis so highest importance appears at top
plt.gca().invert_yaxis()  # gca() = get current axes

# Add subtle vertical grid lines for easier reading
plt.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

### Business Interpretation

The feature importance scores tell us which customer characteristics are most predictive of churn.

**How to read this:**
- Higher importance = stronger predictor of churn
- These features have the most influence on the model's predictions

---

## Critical Insight: Understanding TotalCharges

**You'll notice TotalCharges is ranked as the #1 most important feature. Let's understand what this really means!**

### Why TotalCharges is Powerful

**TotalCharges = MonthlyCharges × TenureMonths**

The model learned a valid and important pattern:

**The Pattern:**
- Customers with **low TotalCharges** are more likely to churn
- Customers with **high TotalCharges** are more likely to stay

**Why This Makes Sense:**
- **Low TotalCharges** indicates new customers who haven't built loyalty yet
- **High TotalCharges** indicates long-term customers who have demonstrated commitment
- TotalCharges combines **both** how long someone has been a customer AND how much they spend

**Business Insight:** Customer loyalty and lifetime value are the strongest predictors of retention!

### The Multicollinearity Issue

However, there's an important statistical consideration:

**TotalCharges and TenureMonths are highly correlated** because TotalCharges is calculated from tenure.

This creates **redundancy** rather than true "data leakage":
- Both features tell a similar story about customer loyalty
- TenureMonths says "how long they've been a customer"
- TotalCharges says "how long they've been a customer AND how much they've spent"

**This isn't problematic for prediction** - both are legitimate indicators of customer commitment!

---

### The Practical Considerations

**For Production Use:**

TotalCharges is actually a **valuable feature** to keep because:
1. It's available at prediction time (every customer has a current TotalCharges)
2. It reflects real loyalty and commitment
3. It combines tenure with spending behavior
4. It's easy to calculate and understand

**The Trade-off:**

While TotalCharges and TenureMonths provide overlapping information, you might choose to:
- **Keep both** - They offer slightly different perspectives (time vs. value)
- **Keep just TotalCharges** - It captures both dimensions in one feature
- **Keep just TenureMonths** - It's more directly interpretable

In practice, **keeping TotalCharges is perfectly valid** for a churn prediction model!

---

### The Other Important Features

Let's look at features **#2-10** that provide additional insights:

1. **TenureMonths (#2):** Direct measure of customer age
   - Newer customers are riskier (actionable: focus on first-year retention programs)

2. **MonthlyCharges (#3):** Current monthly bill amount
   - High charges may indicate value perception issues (actionable: review pricing)

3. **AvgChargesPerMonth (#4):** Average monthly spending rate
   - Complements the tenure/charges relationship

4. **Age (#5):** Customer demographics
   - Different age groups have different needs (actionable: segment by demographics)

5. **ContractType (#6-7):** Month-to-month vs annual contracts
   - Contract length strongly predicts commitment (actionable: incentivize longer contracts)

6. **SupportCalls (#8):** Customer service interactions
   - High support needs may indicate dissatisfaction (actionable: improve product quality)

7. **IsNewCustomer (#9):** Binary flag for new customers
   - Confirms early-stage vulnerability (actionable: enhance onboarding)

### The Lesson for Data Scientists

**Feature importance requires business context!**

When evaluating feature importance, ask:
1. **Does this feature make business sense?** (TotalCharges = customer loyalty - Yes)
2. **Is this information available when making predictions?** (Yes)
3. **Are there redundant features?** (TotalCharges overlaps with TenureMonths)
4. **Can we take action on this insight?** (Focus on early customer retention - Yes)

### Actionable Recommendations

Based on the **top predictive features**, you might recommend:

1. **Create a "First Year Success Program"** - TenureMonths/TotalCharges show new customers are at highest risk
2. **Review pricing strategy** - MonthlyCharges impact value perception
3. **Improve onboarding experience** - IsNewCustomer flag confirms early vulnerability
4. **Offer contract incentives** - ContractType strongly predicts retention
5. **Enhance product quality** - SupportCalls indicate friction points
6. **Segment retention efforts** - Age and demographics matter for targeting

---

# Section 7: Make Predictions on New Customers

Now let's use our best model to predict churn for new customers that TeleConnect just acquired!

## Real-World Scenario

Imagine TeleConnect gets new customer data each week. The retention team needs to:
1. **Identify** high-risk customers
2. **Prioritize** who to contact first
3. **Take action** with targeted retention offers

We'll simulate this by creating a few hypothetical new customers and predicting their churn risk.

**Note:** In production, these would be real customer records that need the same preprocessing as our training data!

In [ ]:
# Select the best performing model based on test accuracy
# In production, you might also consider precision/recall depending on business priorities

# .idxmax() returns the index (row number) of the maximum value
best_model_name = comparison_df.loc[comparison_df['Accuracy'].idxmax(), 'Model']

print(f"Best Model: {best_model_name}")
print(f"   Test Accuracy: {comparison_df['Accuracy'].max():.1%}")
print(f"   Precision: {comparison_df.loc[comparison_df['Accuracy'].idxmax(), 'Precision']:.1%}")
print(f"   Recall: {comparison_df.loc[comparison_df['Accuracy'].idxmax(), 'Recall']:.1%}")

# Assign the best model object to a variable for predictions
# This allows us to use whichever model performed best
if best_model_name == 'Logistic Regression':
    best_model = log_reg
elif best_model_name == 'Decision Tree':
    best_model = decision_tree
else:  # Random Forest
    best_model = random_forest

print(f"\nUsing {best_model_name} for production predictions")

In [ ]:
# Select a sample of test customers to simulate "new" customer scoring
# In production, these would be actual new customers from the database
new_customers = X_test.head(10).copy()  # First 10 test customers
actual_outcomes = y_test.head(10).copy()  # Their actual churn status (for validation)

# Make predictions using the best model
# .predict() returns binary predictions: 0 (stay) or 1 (churn)
churn_predictions = best_model.predict(new_customers)

# .predict_proba() returns probability estimates: [P(stay), P(churn)]
# We take the second column [:, 1] which is P(churn)
# Probabilities range from 0.0 (definitely won't churn) to 1.0 (definitely will churn)
churn_probabilities = best_model.predict_proba(new_customers)[:, 1]

# Create results dataframe for easy viewing
results_df = pd.DataFrame({
    'Customer_ID': range(1001, 1011),  # Simulated customer IDs
    'Churn_Probability': churn_probabilities,  # Risk score (0.0 to 1.0)
    # Convert binary predictions to readable labels
    'Predicted_Churn': ['Yes' if p == 1 else 'No' for p in churn_predictions],
    # Show actual outcomes for validation (wouldn't have this in production)
    'Actual_Churn': ['Yes' if a == 1 else 'No' for a in actual_outcomes],
    # Categorize risk levels using pd.cut() to bin probabilities
    # bins=[0, 0.3, 0.7, 1.0] creates three categories:
    #   0.0-0.3 = Low Risk
    #   0.3-0.7 = Medium Risk
    #   0.7-1.0 = High Risk
    'Risk_Level': pd.cut(churn_probabilities, 
                         bins=[0, 0.3, 0.7, 1.0], 
                         labels=['Low', 'Medium', 'High'])
})

# Sort by churn probability (highest risk first)
# This prioritizes which customers the retention team should contact
results_df = results_df.sort_values('Churn_Probability', ascending=False)

print("\nNEW CUSTOMER CHURN PREDICTIONS")
print("="*85)
print(results_df.to_string(index=False))
print("="*85)

In [ ]:
# Create two visualizations: individual risk scores and risk distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))  # 1 row, 2 columns

# Chart 1: Horizontal bar chart showing churn probability for each customer
# Color-code bars by risk level for quick visual assessment
colors = ['red' if p > 0.7 else 'orange' if p > 0.3 else 'green' 
          for p in results_df['Churn_Probability']]
ax1.barh(results_df['Customer_ID'].astype(str), results_df['Churn_Probability'], color=colors)
ax1.set_xlabel('Churn Probability', fontsize=12, fontweight='bold')
ax1.set_ylabel('Customer ID', fontsize=12, fontweight='bold')
ax1.set_title('Churn Risk by Customer', fontsize=14, fontweight='bold')
# Add vertical line at 50% threshold (common decision boundary)
ax1.axvline(x=0.5, color='black', linestyle='--', alpha=0.5, label='50% threshold')
ax1.legend()
ax1.grid(axis='x', alpha=0.3)

# Chart 2: Pie chart showing distribution of risk levels
# Helps retention team understand overall workload
risk_counts = results_df['Risk_Level'].value_counts()  # Count customers in each category
ax2.pie(risk_counts, labels=risk_counts.index, autopct='%1.0f%%',  # autopct shows percentages
        colors=['red', 'orange', 'green'], startangle=90)  # Start from top (90 degrees)
ax2.set_title('Distribution of Risk Levels', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

### How to Use These Predictions

**Churn Probability Interpretation:**
- **0.0 - 0.3 (Low Risk):** Customer likely to stay - monitor normally
- **0.3 - 0.7 (Medium Risk):** Some concern - consider proactive outreach
- **0.7 - 1.0 (High Risk):** Very likely to churn - immediate intervention needed!

**Recommended Actions by Risk Level:**

**Low Risk (< 30%):**
- No immediate action needed
- Continue regular customer service
- Monitor for changes in behavior

**Medium Risk (30-70%):**
- Send satisfaction survey
- Offer product upgrade information
- Proactive customer service check-in

**High Risk (> 70%):**
- **URGENT:** Personal outreach from retention team
- Offer special retention discount (e.g., 20% off for 6 months)
- Address specific pain points
- Consider contract incentives

### Business Impact

By prioritizing high-risk customers, TeleConnect can:
- **Save money:** Focus retention budget on customers who need it most
- **Improve ROI:** Better conversion rates on retention efforts
- **Increase satisfaction:** Proactive support prevents issues from escalating

---

# Section 8: Save Model for Production Deployment

The final step is to save our best model so it can be used in TeleConnect's production systems.

## What is "Production Deployment"?

In real companies, ML models are deployed into live systems:
- **Customer database** → Feeds customer data to the model
- **Model** → Makes churn predictions
- **CRM system** → Flags high-risk customers for the retention team
- **Dashboard** → Shows real-time churn risk analytics

This happens automatically, often running daily or weekly!

## Saving the Model

We'll save our best model as a `.pkl` file that can be:
- Loaded into production systems
- Used by other data scientists
- Deployed as an API endpoint
- Integrated into automated workflows

Let's save it!

In [ ]:
# Save the best model to disk using pickle
# 'wb' = write binary mode
with open('best_churn_model.pkl', 'wb') as f:
    # pickle.dump() serializes the model object into binary format
    # This preserves the model's learned parameters and structure
    pickle.dump(best_model, f)

print("Model saved successfully!")

print(f"\nSaved as: best_churn_model.pkl")
print(f"   Model type: {best_model_name}")
print(f"   Test accuracy: {comparison_df['Accuracy'].max():.1%}")
# os.path.getsize() returns file size in bytes; divide by 1024 for kilobytes
print(f"   File size: {os.path.getsize('best_churn_model.pkl') / 1024:.1f} KB")

print("\nThis model can now be:")
print("   • Loaded into production systems")
print("   • Used to score new customers automatically")
print("   • Integrated with TeleConnect's CRM")
print("   • Deployed as a prediction API")

### How Production Systems Use This Model

**Weekly Churn Prediction Workflow:**

```
1. Extract customer data from database
   ↓
2. Preprocess data (same steps as Part 1)
   ↓
3. Load saved model
   ↓
4. Generate churn predictions
   ↓
5. Flag high-risk customers in CRM
   ↓
6. Email retention team with priority list
   ↓
7. Update dashboard with new predictions
```

### Model Monitoring & Maintenance

In production, companies also:
- **Monitor performance:** Track if accuracy decreases over time
- **Retrain regularly:** Update model with new data (monthly/quarterly)
- **A/B test:** Compare new model versions against the current one
- **Track business impact:** Measure actual retention improvements

ML models aren't "set it and forget it" - they need ongoing care!

---

# Congratulations! You've Completed Part 2!

## What You Learned

In this lab, you:

- **Established a baseline** to compare against  
- **Trained three different models** (Logistic Regression, Decision Tree, Random Forest)  
- **Evaluated performance** using accuracy, precision, and recall  
- **Checked for overfitting** by comparing training vs. test accuracy  
- **Identified churn drivers** through feature importance analysis  
- **Made predictions** on new customers with risk scoring  
- **Saved a production model** ready for deployment  

---

## Key Takeaways

### 1. Model Selection
Different models have different strengths:
- **Logistic Regression:** Simple, interpretable, fast
- **Decision Tree:** Visual, handles non-linear patterns, can overfit
- **Random Forest:** Most accurate, robust, harder to interpret

### 2. Evaluation Metrics Matter
- **Accuracy:** Overall correctness
- **Precision:** How trustworthy are positive predictions?
- **Recall:** How many actual positives did we catch?
- Choose metrics based on business priorities!

### 3. Overfitting is Real
- Models can memorize training data instead of learning patterns
- Always evaluate on unseen test data
- Big gap between training/test accuracy = red flag!

### 4. Interpretability Creates Value
- Feature importance reveals *why* customers churn
- Understanding drivers enables targeted interventions
- ML isn't just prediction - it's insight!

### 5. Production Requires Planning
- Models need preprocessing pipelines
- Predictions must be actionable
- Ongoing monitoring and retraining are essential

---

## Reflection Questions

Take a few minutes to discuss with your group:

1. Looking at the confusion matrices, which type of error (false positive or false negative) is more costly for TeleConnect's business? Why?

2. Based on the feature importance analysis, what are two specific actions you would recommend to TeleConnect's management team?

3. Why is it important to check for overfitting? What does a large gap between training and test accuracy tell us?

4. How would you explain to a non-technical manager why we need three different metrics (accuracy, precision, recall) instead of just one?

5. What are some limitations of using historical data to predict future customer behavior? When might the model fail?

---

Great work! You now have the foundation to tackle real machine learning problems in business.

---

*Questions? Reach out to your instructor.*